# Deep Reinforcement Learning (Sp25) — Lecture 21: Imitation Learning
**Instructor:** Dr. Mohammad Hossein Rohban  
**Summary by:** Amirhossein Asadi

---

## Overview
Imitation Learning (IL) enables agents to learn effective policies by observing expert demonstrations, bypassing the need for explicit reward functions. This notebook covers:
- The challenge of reward engineering in RL
- Behavioral Cloning (BC) as a supervised learning approach
- The distribution mismatch problem and the DAgger algorithm
- A conceptual setup for Inverse Reinforcement Learning (IRL)

We will use CartPole-v1 as our running example.

## 1. Import Required Libraries
We use PyTorch for neural networks, Gymnasium for the RL environment, and standard Python libraries for data handling and visualization.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gymnasium as gym
import matplotlib.pyplot as plt
from typing import List, Tuple
from collections import deque
import random
import os

# Set random seeds for reproducibility
def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. Define Expert Policy and Environment
We use the CartPole-v1 environment. The expert policy is a simple heuristic: push the cart in the direction of the pole's velocity. This is near-optimal for CartPole and serves as a stand-in for a human or pre-trained expert.

In [ ]:
env = gym.make('CartPole-v1')

# Heuristic expert: push in the direction of pole velocity
class CartPoleExpert:
    def __call__(self, obs: np.ndarray) -> int:
        x, x_dot, theta, theta_dot = obs
        return int(theta_dot > 0)

expert_policy = CartPoleExpert()

## 3. Generate Expert Demonstrations
We collect state-action pairs by running the expert policy in the environment. These will be used to train the behavioral cloning agent.

In [ ]:
def collect_expert_data(env: gym.Env, expert, n_episodes: int = 20) -> Tuple[np.ndarray, np.ndarray]:
    """
    Collect state-action pairs from the expert policy.
    Returns:
        states: (N, obs_dim)
        actions: (N,)
    """
    states = []
    actions = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        while not done:
            action = expert(obs)
            states.append(obs)
            actions.append(action)
            obs, _, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
    return np.array(states), np.array(actions)

expert_states, expert_actions = collect_expert_data(env, expert_policy, n_episodes=40)
print(f"Collected {len(expert_states)} expert state-action pairs.")

## 4. Behavioral Cloning: Supervised Learning Approach
We train a neural network to imitate the expert by mapping states to actions using the collected dataset. This is a supervised learning problem.

In [ ]:
class BCPolicy(nn.Module):
    """Simple MLP for Behavioral Cloning."""
    def __init__(self, obs_dim: int, n_actions: int, hidden_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

def train_bc(policy: nn.Module, states: np.ndarray, actions: np.ndarray, epochs: int = 20, batch_size: int = 64) -> list:
    """Train the BC policy using cross-entropy loss."""
    policy.train()
    optimizer = optim.Adam(policy.parameters(), lr=1e-3)
    losses = []
    n = len(states)
    for epoch in range(epochs):
        perm = np.random.permutation(n)
        epoch_loss = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            batch_states = torch.FloatTensor(states[idx]).to(device)
            batch_actions = torch.LongTensor(actions[idx]).to(device)
            logits = policy(batch_states)
            loss = F.cross_entropy(logits, batch_actions)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(idx)
        losses.append(epoch_loss / n)
    return losses

obs_dim = env.observation_space.shape[0]
n_actions = env.action_space.n
bc_policy = BCPolicy(obs_dim, n_actions).to(device)
bc_losses = train_bc(bc_policy, expert_states, expert_actions, epochs=30)
plt.plot(bc_losses)
plt.title('Behavioral Cloning Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

## 5. Evaluate Behavioral Cloning Policy
We deploy the trained BC policy in the environment and compare its performance to the expert.

In [ ]:
def evaluate_policy(env: gym.Env, policy, n_episodes: int = 10, is_expert: bool = False) -> float:
    """Evaluate a policy (expert or BC) in the environment."""
    returns = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        total_reward = 0.0
        while not done:
            if is_expert:
                action = policy(obs)
            else:
                obs_tensor = torch.FloatTensor(obs).unsqueeze(0).to(device)
                with torch.no_grad():
                    logits = policy(obs_tensor)
                    action = torch.argmax(logits, dim=-1).item()
            obs, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated
        returns.append(total_reward)
    return np.mean(returns)

bc_return = evaluate_policy(env, bc_policy, n_episodes=20)
expert_return = evaluate_policy(env, expert_policy, n_episodes=20, is_expert=True)
print(f"Expert average return: {expert_return:.1f}")
print(f"BC policy average return: {bc_return:.1f}")

## 6. Addressing Distribution Mismatch: DAgger Algorithm
Behavioral Cloning suffers from distribution mismatch: the learned policy may visit states not seen in the expert data, leading to compounding errors. DAgger (Dataset Aggregation) addresses this by iteratively collecting new data from the learned policy and querying the expert for the correct action in those states.

In [ ]:
def dagger_collect(env: gym.Env, policy, expert, n_episodes: int = 10) -> Tuple[np.ndarray, np.ndarray]:
    """
    Run the current policy, collect states, and label with expert actions.
    """
    states = []
    actions = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        while not done:
            obs_tensor = torch.FloatTensor(obs).unsqueeze(0).to(device)
            with torch.no_grad():
                logits = policy(obs_tensor)
                action = torch.argmax(logits, dim=-1).item()
            states.append(obs)
            expert_action = expert(obs)
            actions.append(expert_action)
            obs, _, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
    return np.array(states), np.array(actions)

## 7. Implement DAgger and Aggregate Dataset
We alternate between running the current policy, collecting new states, labeling them with the expert, and retraining the policy on the aggregated dataset.

In [ ]:
dagger_policy = BCPolicy(obs_dim, n_actions).to(device)
# Start with expert data
agg_states = list(expert_states)
agg_actions = list(expert_actions)
dag_losses = []
num_dagger_iters = 10
for it in range(num_dagger_iters):
    # Retrain policy on aggregated data
    states_np = np.array(agg_states)
    actions_np = np.array(agg_actions)
    losses = train_bc(dagger_policy, states_np, actions_np, epochs=5)
    dag_losses.extend(losses)
    # Collect new data using current policy, label with expert
    new_states, new_actions = dagger_collect(env, dagger_policy, expert_policy, n_episodes=5)
    agg_states.extend(new_states)
    agg_actions.extend(new_actions)
plt.plot(dag_losses)
plt.title('DAgger Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

## 8. Evaluate DAgger Policy
We test the final DAgger-trained policy and compare its performance to the expert and the initial BC policy.

In [ ]:
dagger_return = evaluate_policy(env, dagger_policy, n_episodes=20)
print(f"DAgger policy average return: {dagger_return:.1f}")
print(f"Expert average return: {expert_return:.1f}")
print(f"BC policy average return: {bc_return:.1f}")

## 9. Inverse Reinforcement Learning (IRL) Setup (Conceptual Code)
Inverse RL aims to infer the reward function that the expert is optimizing. Below is a conceptual outline for IRL; full implementation is beyond this notebook's scope.


In [ ]:
# Conceptual IRL outline (not executable)
# 1. Collect expert trajectories: (s, a) pairs
# 2. Define a reward function parameterization (e.g., neural network)
# 3. Optimize reward parameters so that the expert's trajectories are more likely under the induced optimal policy than random trajectories
# 4. Use RL to find the optimal policy for the learned reward
# 5. Iterate until convergence

# Example (pseudo-code):
# reward_net = RewardNetwork()
# for irl_iter in range(num_irl_iters):
#     # Step 1: Fit reward_net to distinguish expert vs. learner trajectories
#     # Step 2: RL to optimize policy for current reward_net
#     # Step 3: Collect new learner trajectories
#     pass

print("For full IRL, see algorithms like MaxEnt IRL, AIRL, or GAIL.")